# SIG Spectra Tutorial: From Raw Scans to Clean Curves

This notebook is a **step-by-step walkthrough** for visualizing SVC `.sig` spectra using the helpers shipped in this repository. It is aimed at users who are still getting comfortable with Python — every step is annotated, and we plot the spectra at each stage so you can see *why* each processing step exists.

**What we will cover**
1. Inspecting and plotting a small subset of **raw, unprocessed** `.sig` files — notice the sensor gaps and the trailing junk past the sensor-calibration end-line.
2. Running the **preprocessing step** (`SigFileProcessor`) to truncate each file at its instrument end-line value.
3. Spotting **calibration scans** (spectralon white references) and explaining what to do with them.
4. Running the project's **resampler** to merge the three sensors and smooth onto a 1 nm grid.
5. Detecting and removing **outlier scans** (failed measurements, mis-aimed scans, etc.).
6. Producing a final clean plot of your spectra.

> Companion notebook: see [`sig_spectra_visualization.ipynb`](sig_spectra_visualization.ipynb) for a more compact version of the same workflow.

## 0. Setup

We need three things on our Python path:
- `specdal` — a third-party library for reading `.sig` files into a `Collection` object.
- `pandas`, `numpy`, `matplotlib` — the usual data + plotting stack.
- The local `pipeline/` package shipped with this repo — that is where `SigFileProcessor`, `resample_spectra`, and `SigSpectraAverager` live.

If `specdal` is not installed yet, uncomment the install cell below.

In [ ]:
# !pip install specdal

In [ ]:
import sys
from pathlib import Path

# Make the project package importable from the repo root, notebooks/, or a worktree.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pipeline").is_dir():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing pipeline/.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Project root:", project_root)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from specdal import Collection

# Project helpers
from pipeline.sig_processor import SigFileProcessor
from pipeline.resampler import resample_spectra
from pipeline.processor import SigSpectraAverager

BUILTIN_SENSOR_CALIBRATION = dict(SigFileProcessor.DEFAULT_CORRECTION_TYPES)

%matplotlib inline

### Point at your raw data

Set `RAW_SIG_DIR` to a folder containing **untouched** `.sig` files straight off the SVC instrument. Output folders for the rest of the notebook are derived from `RUN_NAME` so each run gets its own subfolder under `pipeline_outputs/`.

In [ ]:
# CHANGE THESE TWO LINES TO MATCH YOUR DATA -------------------------------
RAW_SIG_DIR = Path.home() / "Downloads" / "a4any_sb_2025-cn_ch-svc-aviris_bottom"
RUN_NAME    = "a4any_sb_aviris_bottom_tutorial"
# -------------------------------------------------------------------------

processed_dir = project_root / "pipeline_outputs" / "sig_processed"  / RUN_NAME
resampled_dir = project_root / "pipeline_outputs" / "sig_resampled" / RUN_NAME
resampled_csv = resampled_dir / f"{RUN_NAME}_merged_spectra.csv"

print("Raw .sig folder:       ", RAW_SIG_DIR, "  exists?", RAW_SIG_DIR.exists())
raw_files = sorted(RAW_SIG_DIR.glob("*.sig"))
print(f"Raw .sig files found:  {len(raw_files)}")

## 1. Peek inside a raw `.sig` file

Before plotting anything, let's open one file in plain Python and just look at it. A `.sig` file is a small text file with two sections:
- A **header** of `key= value` metadata lines (instrument, integration time, GPS, weather, etc.).
- A `data=` marker followed by a four-column table: `wavelength  ref_radiance  tgt_radiance  pct_reflectance`.

The SVC HR-1024i instrument contains **three separate sensors** (Si, InGaAs, extended InGaAs). The `.sig` file stores them back-to-back, so the wavelength column **jumps backward** at every sensor boundary.

Past the last data row, the instrument writes a trailing calibration block. That block is what `SigFileProcessor` will trim away in Step 2.

In [ ]:
sample_path = raw_files[0]
print("File:", sample_path.name, "\n")

with open(sample_path) as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}\n")
print("--- header (first 20 lines) ---")
print("".join(lines[:20]))
print("--- last 5 lines (the trailing block we will trim) ---")
print("".join(lines[-5:]))

## 2. Plot a few raw spectra and notice the sensor gaps

Now load a handful of raw files with `specdal.Collection` and plot them. Watch for:
- **Step discontinuities** near ~1000 nm and ~1900 nm — those are the sensor splice points.
- **Overlap regions** where the same wavelength is measured by two sensors at once.

In [ ]:
raw_collection = Collection(name=f"{RUN_NAME}_raw")
raw_collection.read(directory=str(RAW_SIG_DIR))
print(f"Loaded {len(raw_collection.spectra)} raw spectra")

In [ ]:
# Plot only the first few spectra so the chart stays readable.
SUBSET_SIZE = 5
subset = raw_collection.spectra[:SUBSET_SIZE]

fig, ax = plt.subplots(figsize=(11, 5))
for spectrum in subset:
    m = spectrum.measurement
    ax.plot(m.index, m.values, label=spectrum.name, alpha=0.8)

ax.set_title(f"Raw .sig spectra (first {SUBSET_SIZE} files, no preprocessing yet)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.legend(fontsize="small")
ax.grid(alpha=0.3)
plt.show()

**What you should see**
- Two visible kinks per curve — the sensor splice points.
- Curves may briefly double back on themselves at the boundaries (overlap region).
- These artifacts come from the instrument, not from the surface you scanned. We'll fix them in Step 4 (resampling).

## 3. Preprocessing — truncate at the instrument end-line

The first thing the project's pipeline does is **truncate each raw `.sig` file** at the line that starts with the instrument's end-line wavelength (e.g. `2517.9` for the Silver SVC, `2520.4` for the Bronze). Everything after that line is trailing instrument output that we don't want polluting downstream analysis.

`SigFileProcessor` does this automatically. The class:
1. Reads each `.sig` line by line.
2. Copies lines verbatim into the output file.
3. Stops copying as soon as it sees the end-line value.

Before processing, it's a good idea to verify that **all files in the folder come from the same instrument** — mixing Bronze and Silver data in a single batch is a common cause of silently bad outputs.

In [ ]:
# A 'silver' processor is fine for the consistency check — it doesn't write anything
# yet; we just want the inspection helper.
checker = SigFileProcessor(correction_type="silver")
report = checker.check_instrument_consistency(str(RAW_SIG_DIR))

print(f"Total files:    {report['total_files']}")
print(f"Consistent?     {report['consistent']}")
print(f"Instrument:     {report['instrument']}")
print(f"Instrument name:{report['instrument_name']}")
for w in report['warnings']:
    print("  warning:", w)

### Load the sensor calibration

Here, **sensor calibration** means the instrument-specific end-line table used for trimming `.sig` files. It is separate from the white-reference calibration scans discussed later. The full pipeline can infer a file named `config/calibrations/<input_dir_name>.json`; in this tutorial we load the sensor calibration explicitly so the source of the end-line values is visible.

In [ ]:
# By convention, a run-specific sensor calibration uses the raw folder name.
# If your calibration has a different name, replace this path explicitly.
SENSOR_CALIBRATION_FILE = project_root / "config" / "calibrations" / f"{RAW_SIG_DIR.name}.json"
SigFileProcessor.DEFAULT_CORRECTION_TYPES = dict(BUILTIN_SENSOR_CALIBRATION)

if SENSOR_CALIBRATION_FILE.exists():
    sensor_calibration = SigFileProcessor.load_default_correction_types(SENSOR_CALIBRATION_FILE)
    print("Loaded sensor calibration:", SENSOR_CALIBRATION_FILE)
else:
    sensor_calibration = dict(BUILTIN_SENSOR_CALIBRATION)
    print("No run-specific sensor calibration file found; using built-in defaults.")

pd.Series(sensor_calibration, name="end_line_nm").to_frame()

Pick the matching end-line value from the loaded sensor calibration. We'll match the instrument report from the previous cell.

In [ ]:
sensor_type = report["instrument_name"].lower()
if sensor_type not in sensor_calibration:
    raise RuntimeError(
        f"Unknown instrument '{report['instrument_name']}'. "
        "Add it to the sensor calibration file or pass correction_value= manually."
    )

end_line_value = sensor_calibration[sensor_type]
processor = SigFileProcessor(correction_value=end_line_value)
print(f"Using sensor_type={sensor_type!r}, end_line={end_line_value!r}")

processed_dir.mkdir(parents=True, exist_ok=True)
processor.process_sig_files(
    input_folder=str(RAW_SIG_DIR),
    output_folder=str(processed_dir),
    verbose=False,
)
print(f"Wrote {len(list(processed_dir.glob('*.sig')))} processed files to {processed_dir}")

In [ ]:
# Show that the trailing block really got trimmed: line counts before vs. after.
raw_lines  = sum(1 for _ in open(sample_path))
proc_lines = sum(1 for _ in open(processed_dir / sample_path.name))
print(f"{sample_path.name}")
print(f"  raw       : {raw_lines} lines")
print(f"  processed : {proc_lines} lines")
print(f"  trimmed   : {raw_lines - proc_lines} lines")

Re-plot the same subset using the **processed** files. This plot can look very similar to the raw plot: preprocessing does not smooth the spectra, merge the sensors, or fix the splice kinks. It only removes file rows after the instrument-specific end-line value. The line-count check above is the clearest proof that trimming happened; the visible cleanup comes later during resampling.

In [ ]:
processed_collection = Collection(name=f"{RUN_NAME}_processed")
processed_collection.read(directory=str(processed_dir))

fig, ax = plt.subplots(figsize=(11, 5))
for spectrum in processed_collection.spectra[:SUBSET_SIZE]:
    m = spectrum.measurement
    ax.plot(m.index, m.values, label=spectrum.name, alpha=0.8)

ax.set_title(f"Processed .sig spectra (first {SUBSET_SIZE} files, end-line trimmed)")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.legend(fontsize="small")
ax.grid(alpha=0.3)
plt.show()

## 4. What about calibration scans?

Field SVC scanning typically alternates between two kinds of measurements:
1. **White-reference (calibration) scans** of a Spectralon panel — used by the instrument to convert raw radiance into reflectance. After the file is written, the white reference appears as a near-flat curve with reflectance ≈ 1.0.
2. **Target scans** of the actual sample (leaf, soil, canopy, etc.) — the curves you actually care about.

When you pull every file into one plot, the white references show up as a **flat band of curves clustered near 1.0**, while target scans sit much lower. We usually want to **filter them out** before averaging — otherwise they bias the group statistics. The pipeline does *not* delete them for you so you can audit the run.

In [ ]:
# Pull every spectrum's mean reflectance over 450–900 nm (a clean visible/NIR window).
rows = []
for spectrum in processed_collection.spectra:
    m = spectrum.measurement
    in_window = m[(m.index >= 450) & (m.index <= 900)]
    rows.append({
        "name": spectrum.name,
        "mean_reflectance_450_900": in_window.mean(),
    })

summary = pd.DataFrame(rows)
summary.head(10)

In [ ]:
# Conservative threshold: anything with a mean reflectance above 0.9 in this
# window is almost certainly a Spectralon white reference.
WHITE_REF_THRESHOLD = 0.9

is_calibration = summary["mean_reflectance_450_900"] > WHITE_REF_THRESHOLD
calibration_names = set(summary.loc[is_calibration, "name"].tolist())
target_names = set(summary.loc[~is_calibration, "name"].tolist())

print(f"Calibration (white-reference) scans: {len(calibration_names)}")
print(f"Target scans:                       {len(target_names)}")

In [ ]:
# Plot the two groups side by side so the difference is obvious.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for spectrum in processed_collection.spectra:
    m = spectrum.measurement
    if spectrum.name in calibration_names:
        axes[0].plot(m.index, m.values, color="tab:gray", alpha=0.4)
    else:
        axes[1].plot(m.index, m.values, color="tab:green", alpha=0.3)

axes[0].set_title(f"Calibration scans (n={len(calibration_names)})")
axes[1].set_title(f"Target scans (n={len(target_names)})")
for ax in axes:
    ax.set_xlabel("Wavelength (nm)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Reflectance")
plt.tight_layout()
plt.show()

**Notes on calibration scans**
- You should filter them out (using a threshold like the one above, or a naming convention if your project uses one) before averaging or analysis.
- If a white-reference looks tilted, noisy, or far from 1.0, the panel was likely contaminated or partially shadowed during that calibration — flag the surrounding target scans for review.

## 5. Resample: merge the three sensors and smooth onto a 1 nm grid

The function `pipeline.resampler.resample_spectra` does the heavy lifting:
1. Reads each processed `.sig` file directly (so it understands the multi-sensor layout that specdal does not).
2. Detects the sensor splice points.
3. Applies a linearly-varying multiplicative correction so adjacent sensors agree at the splice.
4. Smooths with a Gaussian whose width matches the local band spacing.
5. Resamples onto an integer 400–2500 nm grid (1 nm steps).

It writes one CSV with one row per sample and one column per wavelength.

In [ ]:
resampled_dir.mkdir(parents=True, exist_ok=True)
merged_path = resample_spectra(
    input_dir=processed_dir,
    output_dir=resampled_dir,
    output_filename=resampled_csv.name,
)
print("Merged spectra CSV:", merged_path)

In [ ]:
merged_df = pd.read_csv(merged_path)
print("Shape:", merged_df.shape)
merged_df.head()

Reuse the calibration filter from Step 4 so we plot only the target scans.

In [ ]:
def plot_wide_spectra(df, sample_col="sample_name", title="Spectra", color=None, alpha=0.4, ax=None):
    """Plot every row of a wide reflectance dataframe.

    Wavelength columns are expected to be numeric or numeric-as-string.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5))

    band_cols = []
    for col in df.columns:
        token = str(col).lstrip("Xx")
        if token.isdigit():
            band_cols.append((int(token), col))
    band_cols.sort()
    wavelengths = [w for w, _ in band_cols]
    ordered = [c for _, c in band_cols]

    for _, row in df.iterrows():
        ax.plot(wavelengths, row[ordered].to_numpy(dtype=float),
                color=color, alpha=alpha)

    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Reflectance")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    return ax

In [ ]:
# specdal strips the .sig extension; the merged CSV keeps it. Normalize so we can
# compare names across both representations.
def normalize(name: str) -> str:
    s = str(name)
    return s[:-4] if s.lower().endswith(".sig") else s

calibration_stems = {normalize(n) for n in calibration_names}
merged_df["is_calibration"] = merged_df["sample_name"].apply(
    lambda s: normalize(s) in calibration_stems
)

targets_df = merged_df.loc[~merged_df["is_calibration"]].copy()
print(f"Target samples after filtering calibration: {len(targets_df)}")

In [ ]:
ax = plot_wide_spectra(
    targets_df.drop(columns=["is_calibration"]),
    title="Resampled target spectra (calibration scans removed)",
    color="tab:green",
    alpha=0.25,
)
plt.show()

Compare this to the plot in Step 2: the sensor steps are gone, the curves are smooth, and every sample sits on the same 400–2500 nm grid. That alignment is what makes any downstream analysis (PCA, ML classification, vegetation indices, etc.) possible.

## 6. Find and remove outlier scans

Even after merging and resampling, individual scans can still be bad: mis-aimed scope, cloud passing over, fibre optic moved during the scan, a leaf that flipped on the panel. We want to spot these before averaging.

A simple, defensible rule:
1. Compute the **median spectrum** across the run — robust to a handful of bad scans.
2. For each sample, compute the **root-mean-square (RMS) distance** from the median.
3. Flag samples whose distance is more than `K` median-absolute-deviations (MAD) above the median distance.

We use MAD instead of standard deviation because a single very bad scan can inflate `std` enough to mask other outliers.

In [ ]:
band_cols = [c for c in targets_df.columns if str(c).lstrip("Xx").isdigit()]
spectra_matrix = targets_df[band_cols].to_numpy(dtype=float)

median_spectrum = np.median(spectra_matrix, axis=0)
rms_distance = np.sqrt(np.mean((spectra_matrix - median_spectrum) ** 2, axis=1))

med = np.median(rms_distance)
mad = np.median(np.abs(rms_distance - med))
K = 4.0  # how many MADs above the median counts as an outlier
threshold = med + K * mad

targets_df = targets_df.assign(rms_distance=rms_distance,
                               is_outlier=rms_distance > threshold)

print(f"Median RMS distance: {med:.4f}")
print(f"MAD:                 {mad:.4f}")
print(f"Threshold (median + {K}·MAD): {threshold:.4f}")
print(f"Flagged outliers:    {int(targets_df['is_outlier'].sum())} / {len(targets_df)}")

targets_df.loc[targets_df["is_outlier"], ["sample_name", "rms_distance"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

kept = targets_df.loc[~targets_df["is_outlier"], ["sample_name"] + band_cols]
flagged = targets_df.loc[targets_df["is_outlier"], ["sample_name"] + band_cols]

plot_wide_spectra(kept, title=f"Kept (n={len(kept)})", color="tab:green", alpha=0.25, ax=axes[0])
plot_wide_spectra(flagged, title=f"Flagged outliers (n={len(flagged)})", color="tab:red", alpha=0.7, ax=axes[1])
plt.tight_layout()
plt.show()

**Notes on outlier handling**
- Tune `K` to your dataset. `K = 3.5–5` is reasonable; smaller values are stricter.
- A flagged scan is **not automatically wrong** — open the original `.sig`, check the field notes, and decide. The flag is a *suggestion*.
- If your run organizes scans into replicates (e.g. five scans per leaf), prefer **median-aggregating each replicate group** instead of dropping individual scans. The `SigSpectraAverager` class does exactly that — see the next cell.

### Optional: median-aggregate replicate scans with `SigSpectraAverager`

If your scan numbering follows `<base>.<NNNN>.sig` and you collected, say, scans 5–7 as three replicates of one target, you can collapse them into a single spectrum with the median:

In [ ]:
# Illustrative — change the group ranges to match your own scan plan.
averager = SigSpectraAverager(merged_df.drop(columns=["is_calibration"]),
                              sample_col="sample_name")

example_groups = [
    (5, 6, 7),     # replicate group 1
    (8, 9, 10),    # replicate group 2
]

aggregated = averager.aggregate(example_groups, method="median")
aggregated.head()

Median-aggregation is naturally outlier-resistant: one bad scan in a group of three barely moves the median, while a mean would be pulled toward it.

## 7. Final clean plot

Putting it all together: calibration scans removed, outliers flagged out, plot the survivors with the median overlaid in black.

In [ ]:
final_df = targets_df.loc[~targets_df["is_outlier"], ["sample_name"] + band_cols]
wavelengths = [int(c) for c in band_cols]
median_curve = final_df[band_cols].median(axis=0).to_numpy(dtype=float)

fig, ax = plt.subplots(figsize=(11, 5))
ax = plot_wide_spectra(final_df, title=f"Clean spectra — {RUN_NAME}",
                       color="tab:green", alpha=0.2, ax=ax)
ax.plot(wavelengths, median_curve, color="black", lw=2, label="Median")
ax.legend()
plt.show()

## Recap

| Step | What we did | Why it matters |
|---|---|---|
| 1 | Peeked at a raw `.sig` file in plain Python | Understand the header / data= layout |
| 2 | Plotted raw spectra with specdal | See the sensor-splice steps |
| 3 | Truncated each file with `SigFileProcessor` | Remove trailing junk past the instrument end-line |
| 4 | Threshold on mean reflectance to find white-reference scans | Stop calibration curves from biasing averages |
| 5 | `resample_spectra` to merge sensors + smooth | One sample per row on a clean 400–2500 nm grid |
| 6 | Median + MAD flag for outliers | Robust to a few very bad scans |
| 7 | Median across kept scans | Final, defensible "typical" curve for the run |

From here you could compute vegetation indices (NDVI, PRI, etc.), feed the spectra into a classifier, or export the cleaned table for downstream modelling.